In [1]:
import os
import requests

import cv2
import duckdb
import numpy as np
import pandas as pd
import sqlalchemy as db
import plotly.express as px
import plotly.graph_objects as go
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import distance
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm 
from sklearn.linear_model import LinearRegression

/home/amos/anaconda3/envs/face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys

sys.path.append(os.path.abspath(os.path.join('../..')))

from utils import *

In [3]:
# if not os.path.exists('utils.py'):
#     url = "https://raw.githubusercontent.com/your-username/CineFace/main/utils.py"
#     with open("utils.py", "wb") as f:
#         f.write(requests.get(url).content)

# import utils
# utils.install_dependencies() # Handles pip install duckdb, huggingface_hub, etc.
# con = utils.get_db_connection()

In [4]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

# Martin Scorsese -- Master of Ensemble Framing

Scorsese presents an interesting case. On a couple metrics, Scorsese is not only not exceptional, but is almost completely average. This is particularly evident when we look at the relationship between the average face size and the average variation in face size, as seen below.

In [5]:
name = "Martin Scorsese"

In [6]:
df = pd.read_sql_query(f"SELECT * FROM vwWorksByDirector WHERE kind = 'movie' AND person_name = '{name}'", conn)
df

,person_id,person_name,title,year,kind,runtime,fact_id,imdb_id,work_id,avg_size,...,z_v_f_per_fr,z_v_f_per_fr_g,z_vert,z_vert_g,z_v_size,z_v_size_g,z_gini,z_gini_g,z_h_spread,z_h_spread_g
0,1032,Martin Scorsese,Raging Bull,1980,movie,129,6205,81398,37493,None,...,0.071882,0.333826,-0.733235,-0.679504,None,None,-1.543230,-1.788850,0.242068,0.167322
1,1032,Martin Scorsese,The King of Comedy,1982,movie,109,6022,85794,37588,None,...,0.240545,0.712659,0.505913,0.256007,None,None,-0.315140,-0.713773,0.237450,0.265478
2,1032,Martin Scorsese,After Hours,1985,movie,97,6091,88680,37639,None,...,-1.190040,-0.931533,0.134673,-0.079175,None,None,0.446091,-0.036726,-0.809361,-0.242829
3,1032,Martin Scorsese,The Color of Money,1986,movie,119,6932,90863,37669,None,...,0.765336,1.289020,-0.046044,-0.134205,None,None,-1.217890,-1.772980,0.494967,0.587989
4,1032,Martin Scorsese,The Last Temptation of Christ,1988,movie,164,6436,95497,37756,None,...,0.802316,1.497690,-0.462878,-0.549433,None,None,-1.213010,-1.558760,0.506919,0.552934
5,1032,Martin Scorsese,GoodFellas,1990,movie,145,6734,99685,37848,None,...,0.502592,0.451908,0.014262,0.195974,None,None,-1.116960,-1.077420,0.349035,0.342600
6,1032,Martin Scorsese,The Age of Innocence,1993,movie,139,7907,106226,38101,None,...,0.623769,0.675357,-0.953611,-1.059710,None,None,-0.999524,-0.943864,0.798977,0.644078
7,1032,Martin Scorsese,Casino,1995,movie,178,7862,112641,38328,None,...,0.904846,0.369675,0.043283,-0.214250,None,None,-1.674590,-1.248010,0.442976,0.030606
8,1032,Martin Scorsese,Gangs of New York,2002,movie,167,6153,217505,38976,None,...,2.518500,2.470690,-1.092490,-1.114740,None,None,-2.307930,-1.906540,0.739699,0.409206
9,1032,Martin Scorsese,The Aviator,2004,movie,170,7251,338751,39095,None,...,2.535280,3.007450,-0.285974,-0.239262,None,None,-2.590090,-1.622230,1.223370,0.559945


In [7]:
g = pd.read_sql_query("SELECT * FROM vwDirectorStats WHERE cnt > 5", conn)
g

,person_id,name,cnt,z_size_mean,z_size_g_mean,z_size_std,z_size_g_std,z_v_size_mean,z_v_size_g_mean,z_v_size_std,...,z_v_dist_g_std,z_v_dist_std,z_vert_mean,z_vert_g_mean,z_vert_g_std,z_vert_std,z_h_spread_mean,z_h_spread_g_mean,z_h_spread_g_std,z_h_spread_std
0,100036,D.W. Griffith,21,None,None,None,None,None,None,None,...,None,None,-0.156232,0.774387,0.597136,0.716038,-0.127035,-1.057120,1.076980,1.019446
1,8636,Cecil B. DeMille,19,None,None,None,None,None,None,None,...,None,None,-1.322948,-0.487030,0.761019,0.852458,0.488785,-0.228991,0.785583,0.648082
2,72061,Frank Lloyd,19,None,None,None,None,None,None,None,...,None,None,0.161396,0.618837,0.743220,0.744338,0.158505,-0.216629,0.635197,0.572529
3,90375,Christy Cabanne,18,None,None,None,None,None,None,None,...,None,None,0.279248,0.648722,0.743836,0.884165,0.088149,-0.052944,1.001909,1.063252
4,42060,Sidney Franklin,10,None,None,None,None,None,None,None,...,None,None,-0.176508,0.657227,0.729082,0.851260,-0.011420,-0.918700,0.847560,0.886634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312,1188853,Georg af Klercker,6,None,None,None,None,None,None,None,...,None,None,-0.174650,0.692913,0.635942,0.683776,-0.308918,-1.253014,0.432225,0.449480
313,6817,Agnès Varda,8,None,None,None,None,None,None,None,...,None,None,-0.425123,-0.473142,0.857096,0.944775,-0.173187,-0.123639,0.857805,0.978684
314,510,Tim Burton,6,None,None,None,None,None,None,None,...,None,None,-0.548889,-0.797902,0.613365,0.737814,-0.237282,-0.215368,0.372528,0.636926
315,21684,Bong Joon Ho,8,None,None,None,None,None,None,None,...,None,None,-0.168296,-0.389346,0.609831,0.769067,-0.453476,-0.072809,0.174214,0.496192


## Avg. Size vs. Size Variance

### Global

In [8]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_directors_against_sample(g, x, y, [name])

## Yearly

In [9]:
x = "z_size"
y = "z_v_size"
plot_director_films(df, x, y, name)

In [10]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_directors_against_sample(g, x, y, [name])

In [11]:
x = "z_size_std"
y = "z_v_size_std"
plot_directors_against_sample(g, x, y, [name])

In [12]:
x = "z_f_per_fr_g_mean"
y = "z_gini_g_mean"
plot_directors_against_sample(g, x, y, [name])

In [13]:
create_gridmap_from_director("Martin Scorsese", engine)

In [15]:
compare_director_to_sample_grid("Martin Scorsese", engine)

In [24]:
create_gridmap_from_director("Steven Spielberg", engine)

In [25]:
compare_director_to_sample_grid("Steven Spielberg", engine)